# `results.html` table to LaTeX

`html/results.html`에 들어 있는 표를 찾아, 논문에 바로 붙여넣을 수 있는 LaTeX 코드로 변환합니다. `TABLE_INDEX`만 바꾸면 원하는 표를 선택할 수 있습니다.

생성된 코드는 `booktabs`, `multirow`, `graphicx` 패키지를 사용하며, 오른쪽에 배치되는 reference-ligand similarity 표에는 `wrapfig`도 필요합니다. HTML의 `rowspan`/`colspan`, 평균 ± 표준편차, 위·아래 화살표, 특수문자, best/second 강조도 함께 변환합니다. `TABLE_INDEX = 0` (binding affinity)은 논문용 전용 렌더러(`binding_affinity_to_latex`)로 출력합니다: `Sequence`/`Structure`/`Density` modality 그룹, `Pre.` 열, leaked(Nesso-1) `$\dagger$` 표시 및 순위 제외, ProFSA와 Ours-C 사이 `\cmidrule`을 스펙에 고정하고, 각 셀의 mean±std(`\std{...}` 매크로)와 best(bold)/next(underline) 강조는 `results.html`에서 매번 재계산합니다. Test = 첫 tier, CL3 filtered = 마지막 tier 열을 사용합니다.

## Configuration

입력 HTML, 출력할 표 번호, 태그 및 크기 옵션을 설정합니다.

In [2]:
from dataclasses import dataclass
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path
import re

from bs4 import BeautifulSoup, NavigableString, Tag

# 노트북을 저장소 루트 또는 notebook/ 디렉터리에서 실행해도 동작합니다.
HTML_PATH = Path("html/results.html")
TABLE_INDEX = 5       # 0 = binding-affinity (Pre. 열 포함); 아래 목록에서 원하는 표 번호로 변경
KEEP_BADGES = True    # Table 1 외 표에서 태그를 유지할지 여부
TABLE1_EXCLUDE_TAGS = True
RESIZE_WIDE = True    # 넓은 표에 \resizebox{.98\textwidth}{!}{...} 적용

# LaTeX 밀도(Density) 모델 표시: 여러 CDG 변형(CDG v2 / v3 …) 중 어느 것을 "Ours (CDG)"
# 행으로 쓸지 고릅니다. HTML method 키 접두가 f"CDG {CDG_VARIANT}" (예: "CDG v2")가 되도록
# 뒤 suffix만 바꾸면 됩니다. (표시명은 항상 'Ours (CDG)'로 고정)
CDG_VARIANT = "v2"

# Table 1에서는 원본 Test와 최종 CL3 filtered 결과만 출력합니다.
# 각 튜플은 HTML 원본에서 그룹이 시작하는 열과 출력할 컬럼명입니다.
TABLE1_GROUP_HEADERS = [
    (2, "Test (N=1312)"),
    (11, "CL3 filtered (N=733)"),
]
TABLE1_CAPTION = (
    r"\textbf{LP-PDBBind binding-affinity prediction} on the original and CL3-filtered "
    r"test sets. \textbf{Bold} marks the best mean and results whose means fall within "
    r"each other's standard-deviation intervals; \underline{underlining} marks the next-best "
    r"mean outside that group."
)

# Similarity performance table: 기존 LP-PDBBind 열은 Table 1과 중복되어 제외합니다.
SIMILARITY_GROUP_HEADERS = [
    (8, "Similarity < 30% (N=453)"),
    (5, "Similarity < 60% (N=813)"),
]
SIMILARITY_EXCLUDE_TAGS = True
REFERENCE_SIMILARITY_EXCLUDE_TAGS = True
SIMILARITY_CAPTION = (
    r"\textbf{LP-PDBBind similarity-stratified binding-affinity prediction} on "
    r"protein-similarity-filtered test sets. \textbf{Bold} marks the best "
    r"mean and results whose means fall within each other's standard-deviation intervals; "
    r"\underline{underlining} marks the next-best mean outside that group."
)


## LaTeX converter

HTML 표 구조와 표별 서식을 LaTeX로 변환하는 함수들입니다.

In [3]:
LINEBREAK = "\ue000"


def resolve_html_path(path: Path) -> Path:
    candidates = [path, Path("notebook") / path]
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    tried = ", ".join(str(p.resolve()) for p in candidates)
    raise FileNotFoundError(f"results.html을 찾지 못했습니다. 확인한 경로: {tried}")


def escape_latex_text(text: str) -> str:
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
        "<": r"$<$",
        ">": r"$>$",
    }
    text = "".join(replacements.get(char, char) for char in text)
    unicode_replacements = {
        "\xa0": " ",
        "±": r"$\pm$",
        "ρ": r"$\rho$",
        "σ": r"$\sigma$",
        "Δ": r"$\Delta$",
        "↓": r"$\downarrow$",
        "↑": r"$\uparrow$",
        "→": r"$\rightarrow$",
        "←": r"$\leftarrow$",
        "≤": r"$\leq$",
        "≥": r"$\geq$",
        "≈": r"$\approx$",
        "×": r"$\times$",
        "·": r"$\cdot$",
        "−": "-",
        "—": "---",
        "–": "--",
        "²": r"\textsuperscript{2}",
        "†": r"\textsuperscript{$\dagger$}",
        "\u200a": " ",
    }
    for old, new in unicode_replacements.items():
        text = text.replace(old, new)
    return text


def inline_to_latex(node, keep_badges: bool = True) -> str:
    if isinstance(node, NavigableString):
        return escape_latex_text(str(node))
    if not isinstance(node, Tag) or node.name in {"script", "style"}:
        return ""
    if node.name == "br":
        return LINEBREAK

    content = "".join(inline_to_latex(child, keep_badges) for child in node.children)
    classes = set(node.get("class", []))

    if "tag" in classes:
        if not keep_badges or not content.strip():
            return ""
        return rf"\,\textsuperscript{{\scriptsize {content.strip()}}}"
    if "nsub" in classes:
        return LINEBREAK + content
    if node.name in {"b", "strong"}:
        return rf"\textbf{{{content}}}"
    if node.name in {"i", "em"}:
        return rf"\textit{{{content}}}"
    if node.name == "sub":
        return rf"\textsubscript{{{content}}}"
    if node.name == "sup":
        return rf"\textsuperscript{{{content}}}"
    if "display:block" in node.get("style", "").replace(" ", "").lower():
        return LINEBREAK + content
    return content


def cell_to_latex(
    cell: Tag, keep_badges: bool = True, preserve_emphasis: bool = True
) -> str:
    raw_parts = inline_to_latex(cell, keep_badges).split(LINEBREAK)
    parts = [re.sub(r"\s+", " ", part).strip() for part in raw_parts]
    parts = [part for part in parts if part]
    if not parts:
        return ""

    classes = set(cell.get("class", []))
    style = cell.get("style", "").replace(" ", "").lower()
    is_bold = cell.name == "th" or "best" in classes or bool(re.search(r"font-weight:(?:[7-9]00|bold)", style))
    is_underlined = "second" in classes or "text-decoration:underline" in style
    if preserve_emphasis and is_bold:
        parts = [rf"\textbf{{{part}}}" for part in parts]
    if preserve_emphasis and is_underlined:
        parts = [rf"\underline{{{part}}}" for part in parts]

    if len(parts) == 1:
        return parts[0]
    align = "c" if cell.name == "th" or "metric" in classes else "l"
    return rf"\shortstack[{align}]{{" + r" \\ ".join(parts) + "}"


@dataclass(frozen=True)
class CellPlacement:
    node: Tag
    row: int
    col: int
    rowspan: int
    colspan: int


def build_layout(table: Tag):
    rows = table.find_all("tr")
    occupied = {}
    placements = []
    ncols = 0

    for row_index, row in enumerate(rows):
        col_index = 0
        for cell in row.find_all(["th", "td"], recursive=False):
            while (row_index, col_index) in occupied:
                col_index += 1
            rowspan = max(1, int(cell.get("rowspan", 1)))
            colspan = max(1, int(cell.get("colspan", 1)))
            placement = CellPlacement(cell, row_index, col_index, rowspan, colspan)
            placements.append(placement)
            for rr in range(row_index, row_index + rowspan):
                for cc in range(col_index, col_index + colspan):
                    occupied[(rr, cc)] = placement
            col_index += colspan
        ncols = max(ncols, col_index)
    return rows, occupied, placements, ncols


def table_title(table: Tag, index: int) -> str:
    title_node = table.find_previous("p", class_="table-title")
    if title_node is not None:
        return re.sub(r"\s+", " ", title_node.get_text(" ", strip=True))
    heading = table.find_previous(["h1", "h2", "h3", "h4"])
    return heading.get_text(" ", strip=True) if heading else f"Results table {index + 1}"


def caption_from_title(title: str) -> str:
    caption = re.sub(r"^Table\s+[A-Za-z0-9.]+\s*[·:—-]\s*", "", title, flags=re.IGNORECASE)
    return escape_latex_text(caption)


def label_from_title(title: str, index: int) -> str:
    match = re.match(r"^Table\s+([A-Za-z0-9.]+)", title, flags=re.IGNORECASE)
    stem = match.group(1) if match else str(index + 1)
    stem = re.sub(r"[^a-z0-9]+", "-", stem.lower()).strip("-")
    return f"tab:results-{stem}"


def infer_alignments(placements, ncols: int) -> str:
    alignments = []
    body_cells = [p for p in placements if p.node.name == "td"]
    for col in range(ncols):
        candidates = [p.node for p in body_cells if p.col <= col < p.col + p.colspan]
        cell = candidates[0] if candidates else None
        if cell is None:
            alignments.append("c")
            continue
        classes = set(cell.get("class", []))
        style = cell.get("style", "").replace(" ", "").lower()
        if "text-align:right" in style:
            alignments.append("r")
        elif "metric" in classes or "text-align:center" in style:
            alignments.append("c")
        else:
            alignments.append("l")
    return "".join(alignments)


def parse_mean_std(cell: Tag):
    value_node = cell.find(class_="val")
    if value_node is None:
        return None
    number = r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)"
    value_match = re.search(number, value_node.get_text(strip=True).replace("−", "-"))
    if value_match is None:
        return None
    sd_node = cell.find(class_="sd")
    sd_match = re.search(number, sd_node.get_text(strip=True)) if sd_node else None
    mean = float(value_match.group())
    std = abs(float(sd_match.group())) if sd_match else 0.0
    return mean, std


def apply_std_macro(value: str) -> str:
    """Replace an inline standard deviation with the LaTeX std macro."""
    pattern = r"\s*\$\\pm\$\s*([0-9]+(?:\.[0-9]+)?)"
    return re.sub(pattern, lambda match: rf"\std{{{match.group(1)}}}", value)


def method_name_to_latex(cell: Tag) -> str:
    """Keep only the method name, dropping badges and detail/note spans."""
    parts = []
    for child in cell.children:
        if isinstance(child, NavigableString):
            parts.append(escape_latex_text(str(child)))
        elif isinstance(child, Tag) and child.name in {"b", "strong", "i", "em"}:
            parts.append(inline_to_latex(child, keep_badges=False))
    return re.sub(r"\s+", " ", "".join(parts)).strip()


def format_two_decimal_places(value: str) -> str:
    """Use two decimal places, padding missing precision invisibly."""
    value = value.strip()
    match = re.fullmatch(r"([+-]?\d+)(?:\.(\d+))?", value)
    if match is None:
        return value
    decimals = match.group(2) or ""
    if len(decimals) > 2:
        rounded = Decimal(value).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
        return f"{rounded:.2f}"
    if len(decimals) == 2:
        return value
    if len(decimals) == 1:
        return value + r"\phantom{0}"
    return value + r".\phantom{0}\phantom{0}"


def format_pair_two_decimal_places(value: str) -> str:
    return " / ".join(format_two_decimal_places(part) for part in value.split(" / "))


def format_similarity_two_decimal_places(value: str) -> str:
    if " / " in value:
        return format_pair_two_decimal_places(value)
    if value.endswith(r"\%"):
        return format_two_decimal_places(value[:-2]) + r"\%"
    return format_two_decimal_places(value)


def table1_metric_emphasis(rows, placements, metric_columns):
    """Rank cells, requiring mutual mean inclusion for the bold tier."""
    values_by_column = {col: [] for col in metric_columns}
    for placement in placements:
        if placement.col not in values_by_column or placement.node.name != "td":
            continue
        row = rows[placement.row]
        if row.select_one(".tag.leaked, .cat-leaked") is not None:
            continue
        parsed = parse_mean_std(placement.node)
        if parsed is not None:
            values_by_column[placement.col].append((placement.row, *parsed))

    emphasis = {}
    for col, values in values_by_column.items():
        if not values:
            continue
        maximize = (col - 2) % 3 != 2  # Pearson/Spearman ↑, RMSE ↓
        best_mean = (max if maximize else min)(mean for _, mean, _ in values)
        best_std = max(std for _, mean, std in values if abs(mean - best_mean) < 1e-12)
        sota_rows = set()
        for row, mean, std in values:
            mean_gap = abs(mean - best_mean)
            if mean_gap <= best_std and mean_gap <= std:
                emphasis[(row, col)] = "bold"
                sota_rows.add(row)

        remaining = [(row, mean) for row, mean, _ in values if row not in sota_rows]
        if remaining:
            second_mean = (max if maximize else min)(mean for _, mean in remaining)
            for row, mean in remaining:
                if abs(mean - second_mean) < 1e-12:
                    emphasis[(row, col)] = "underline"
    return emphasis


def de_novo_to_latex(table: Tag) -> str:
    """Structure-based drug design (VoxBind) table. Row order, method naming, commented
    rows and placeholders follow a fixed spec; all VALUES are pulled live from the de novo
    CrossDocked table in results.html (Score/Min/Dock/High aff. = Avg / Med; QED/SA/Div = Avg)."""
    rows = [r for r in table.select("tbody > tr")
            if len(r.find_all(["th", "td"], recursive=False)) == 19]

    def find(pred):
        for r in rows:
            cells = r.find_all(["th", "td"], recursive=False)
            if pred(cells[0].get_text(" ", strip=True)):
                return cells
        raise ValueError("de novo row not found for spec predicate")

    def clean(cell):
        t = cell.get_text(strip=True).replace("−", "-").replace("—", "---")
        return t if t else "---"

    def vals(cells):  # 7 template columns: Score/Min/Dock/High (Avg / Med) + QED/SA/Div (Avg)
        pair = lambda a, b: f"{clean(cells[a])} / {clean(cells[b])}"
        return [pair(1, 2), pair(3, 4), pair(5, 6), pair(7, 8),
                clean(cells[9]), clean(cells[11]), clean(cells[13])]

    def row(display, pred, comment=False):
        line = " & ".join([display, *vals(find(pred))]) + r" \\"
        return ("% " + line) if comment else line

    body_rows = [
        row("Reference", lambda t: t.startswith("Reference")),
        r"\midrule",
        row("AR", lambda t: t.startswith("AR ")),
        row("Pocket2Mol", lambda t: t.startswith("Pocket2Mol")),
        row("DiffSBDD", lambda t: t.startswith("DiffSBDD")),
        row("TargetDiff", lambda t: t.startswith("TargetDiff")),
        row(r"DecompDiff\textsuperscript{$\dagger$}",
            lambda t: t.startswith("DecompDiff†"), comment=True),
        row(r"VoxBind\textsubscript{\scriptsize $\sigma$=0.9}",
            lambda t: t.startswith("VoxBind σ=0.9"), comment=True),
        row(r"VoxBind\textsubscript{\scriptsize $\sigma$=1.0}",
            lambda t: t.startswith("VoxBind σ=1.0"), comment=True),
        r"% \midrule",
        row(r"DecompDiff\textsubscript{\scriptsize ref-informed}",
            lambda t: t.startswith("DecompDiff reproduced ref-informed")),
        row(r"DecompDiff\textsubscript{\scriptsize ref-free}",
            lambda t: t.startswith("DecompDiff reproduced ref-free")),
        row(r"VoxBind\textsubscript{\scriptsize $\sigma$=0.9}",
            lambda t: t.startswith("VoxBind reproduced") and "σ = 0.9" in t),
        row(r"VoxBind\textsubscript{\scriptsize $\sigma$=1.0}",
            lambda t: t.startswith("VoxBind reproduced") and "σ = 1.0" in t, comment=True),
        r"Funcbind \\",
        r"\midrule",
        r"\textbf{VoxBind + C} & \multicolumn{7}{c}{?} \\",
        row(r"\textbf{VoxBind + CDG}", lambda t: t.startswith("Ours")),
    ]

    caption = (
        r"\textbf{Structure-based drug design results} on 92 test pockets with experimental density data available of CrossDocked2020 benchmark. Vina and "
        r"high-affinity cells report mean / median; arrows indicate the preferred direction."
    )
    body = [
        r"\begin{table}[!t]",
        r"    \centering",
        r"    \caption{",
        f"        {caption}",
        r"    }",
        r"    \label{tab:result-drug-design}",
        r"    \resizebox{.98\textwidth}{!}{%",
        r"        \begin{tabular}{@{}lcccccccc@{}}",
        r"            \toprule",
        r"            \multirow{2}{*}{\textbf{Method}} & \multicolumn{4}{c}{\textbf{Vina evaluation}} & \multicolumn{4}{c}{\textbf{Sample quality}} \\",
        r"            \cmidrule(lr){2-5}\cmidrule(lr){6-9}",
        r"            & \textbf{Score $\downarrow$} & \textbf{Min $\downarrow$} & \textbf{Dock $\downarrow$} & \textbf{High aff. $\uparrow$} & \textbf{QED $\uparrow$} & \textbf{SA $\uparrow$} & \textbf{Diversity $\uparrow$} & \textbf{\# Atoms / Mol} \\",
        r"            \midrule",
    ]
    body.extend("            " + r for r in body_rows)
    body.extend([
        r"            \bottomrule",
        r"        \end{tabular}",
        r"    }",
        r"\end{table}",
    ])
    return "\n".join(body)


def reference_similarity_to_latex(table: Tag) -> str:
    """Transpose reference similarity and render it as a right-side wraptable."""
    methods = []
    values_by_method = []
    for row in table.select("tbody > tr"):
        cells = row.find_all(["th", "td"], recursive=False)
        if len(cells) < 8:
            continue
        method = method_name_to_latex(cells[0])
        bold_match = re.fullmatch(r"\\textbf\{([^{}]+)\}", method)
        methods.append(bold_match.group(1) if bold_match else method)
        values_by_method.append([
            format_similarity_two_decimal_places(cell_to_latex(cell, keep_badges=False))
            for cell in cells[1:8]
        ])

    metric_names = ["Morgan", "Scaffold match", "3D shape", "MACCS", "AtomPair", "RDKit", "Dice"]
    latex_rows = []
    for metric_index, metric in enumerate(metric_names):
        values = [method_values[metric_index] for method_values in values_by_method]
        latex_rows.append(" & ".join([metric, *values]) + r" \\ ".rstrip())

    header = " & ".join([r"\textbf{Metric}", *[rf"\textbf{{{method}}}" for method in methods]]) + " " + chr(92) * 2
    body = [
        r"\begin{wraptable}{r}{0.55\textwidth}",
        r"    \centering",
        r"    \caption{",
        r"        \textbf{Reference-ligand similarity} on the CrossDocked benchmark. Paired similarity values report mean / max; Scaffold match reports the exact-match rate.",
        r"    }",
        r"    \label{tab:results-reference-ligand-similarity}",
        r"    \resizebox{.98\linewidth}{!}{%",
        r"        \begin{tabular}{@{}lccc@{}}",
        r"            \toprule",
        "            " + header,
        r"            \midrule",
    ]
    body.extend("            " + line for line in latex_rows)
    body.extend([
        r"            \bottomrule",
        r"        \end{tabular}",
        r"    }",
        r"\end{wraptable}",
    ])
    return "\n".join(body)


# --- Binding-affinity Table 1 renderer (spec-driven; numbers synced from HTML) ---
# 구조/캡션/Pre./정렬/dagger는 아래 스펙에 고정하고, 각 셀의 mean±std와 bold/underline
# 강조는 results.html에서 매번 재계산합니다. IPNet(frozen)은 leaked($\dagger$, 순위 제외), IPNet(scratch)는 일반 baseline으로 포함하고, Nesso-1도 leaked라
# $\dagger$를 붙이고 순위 산정에서 제외합니다.

# (표시명, HTML method-text 접두, Pre. 마크, leaked/dagger 여부)
BINDING_MODALITY_GROUPS = [
    ("Sequence", [
        ("DeepDTA",              "DeepDTA",           r"\xmark", False),
        ("MolTrans",             "MolTrans",          r"\xmark", False),
        ("HonestAffinity",       "HonestAffinity",    r"\cmark", False),
        (r"Nesso-1$^\dagger$",   "Nesso-1",           r"\cmark", True),
    ]),
    ("Structure", [
        ("HBGSA",                "HBGSA",             r"\xmark", False),
        ("EGNN",                 "EGNN supervised",   r"\xmark", False),
        ("EGNN + TargetDiff",    "EGNN + TargetDiff", r"\xmark", False),
        ("GET",                  "GET",               r"\xmark", False),
        ("CheapNet",             "CheapNet",          r"\xmark", False),
        ("IPNet",                "IPNet (scratch)",   r"\xmark", False),
        (r"IPNet$^\dagger$",     "IPNet (frozen)",    r"\cmark", True),
        ("AEV-PLIG",             "AEV-PLIG",          r"\xmark", False),
        ("DSMBind",              "DSMBind",           r"\cmark", False),
        ("BindNet",              "BindNet",           r"\cmark", False),
        ("ProFSA",               "ProFSA",            r"\cmark", False),
        (r"Ours (C)",             "C pretrained",      r"\cmark", False),
    ]),
    ("Density", [
        # (우선 LaTeX 제외) ("CDG (mask050)", "C+D+G pretrained", r"\cmark", False),
        # (우선 LaTeX 제외) ("CDG + corr", "C+D+G +corr", r"\cmark", False),
        # 여러 CDG 변형 중 CDG_VARIANT(예: v2) 하나만 "Ours (CDG)"로 표시.
        (r"Ours (CDG)",          f"CDG {CDG_VARIANT}", r"\cmark", False),
    ]),
]
# 이 표시명 앞에 \cmidrule(lr){2-9} (baseline vs ours 구분)
BINDING_CMIDRULE_BEFORE = {"Ours (C)"}
BINDING_GROUP_HEADERS = ("Test (N=1312)", "CL3 filtered Test (N=733)")
BINDING_CAPTION = (
    r"\textbf{LP-PDBBind binding-affinity prediction on the \textit{test} and "
    r"\textit{CL3-filtered test} sets.} Metrics are reported as the mean and standard "
    r"deviation over five seeds. \textbf{Bold} marks the best and \underline{underlining} "
    r"marks the next-best; methods whose intervals overlap are marked together. $\dagger$ "
    r"denotes methods trained on leaked data overlapping with the test set; these are "
    r"excluded from marking as they are not comparable to leakage-controlled methods. "
    r"N/A denotes unavailable metrics, and \textbf{Pre.} denotes whether models use "
    r"pre-training on external data."
)


SIMILARITY_MODALITY_CAPTION = (
    r"\textbf{LP-PDBBind similarity-stratified binding-affinity prediction on the "
    r"\textit{CL3-filtered test} set, split by maximum protein-sequence identity to the "
    r"CL3 training set.} Metrics are reported as the mean and standard deviation over five "
    r"seeds. \textbf{Bold} marks the best and \underline{underlining} marks the next-best; "
    r"methods whose intervals overlap are marked together. $\dagger$ denotes methods "
    r"trained on leaked data overlapping with the test set; these are excluded from marking "
    r"as they are not comparable to leakage-controlled methods. N/A denotes unavailable "
    r"metrics, and \textbf{Pre.} denotes whether models use pre-training on external data."
)


BINDING_CL1_CL2_CAPTION = (
    r"\textbf{LP-PDBBind binding-affinity prediction on the \textit{CL1-filtered} and "
    r"\textit{CL2-filtered} test sets.} Metrics are reported as the mean and standard "
    r"deviation over five seeds. \textbf{Bold} marks the best and \underline{underlining} "
    r"marks the next-best; methods whose intervals overlap are marked together. $\dagger$ "
    r"denotes methods trained on leaked data overlapping with the test set; these are "
    r"excluded from marking as they are not comparable to leakage-controlled methods. "
    r"N/A denotes unavailable metrics, and \textbf{Pre.} denotes whether models use "
    r"pre-training on external data."
)


def _binding_parse_metric(cell: Tag):
    tbd = cell.find(class_="tbd")
    if tbd is not None and tbd.get_text(strip=True).lower() in ("n/a", "na"):
        return (None, None)               # explicit not-applicable (zero-shot energy RMSE) → N/A
    val = cell.find(class_="val")
    if val is None:                       # TBA / 빈 셀 → 값 없음
        return None
    sd = cell.find(class_="sd")
    number = r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)"
    mean = float(re.search(number, val.get_text(strip=True).replace("−", "-")).group())
    std = abs(float(re.search(number, sd.get_text(strip=True)).group())) if sd else 0.0
    return mean, std


def _modality_grouped_to_latex(table, metric_indices, group_headers, caption, label, expect_metrics=12, casf_clean=frozenset()):
    # 공통 렌더러: modality 그룹 + Pre. 열 + dagger + bold/underline (binding / similarity 공용)
    html_rows = []
    for row in table.find_all("tr"):
        method_cell = row.find("td", class_="col-method", recursive=False)
        metrics = row.find_all("td", class_="metric", recursive=False)
        if method_cell is None or len(metrics) != expect_metrics:
            continue
        text = method_cell.get_text(" ", strip=True)
        vals = [_binding_parse_metric(metrics[i]) for i in metric_indices]
        html_rows.append((text, vals))

    resolved = []  # (표시명, pre, leaked, [6 (mean,std)])
    for _, methods in BINDING_MODALITY_GROUPS:
        for display, key, pre, leaked in methods:
            match = next((v for text, v in html_rows if text.startswith(key)), None)
            if match is None:
                raise ValueError(f"results.html에서 {key!r}로 시작하는 행을 찾지 못했습니다.")
            disp_e, leaked_e = ((display.replace(r"$^\dagger$", ""), False)
                               if key in casf_clean else (display, leaked))
            resolved.append((disp_e, pre, leaked_e, match))

    # 열별 강조: leaked 행은 순위에서 제외
    maximize = [True, True, False] * len(group_headers)  # r↑ ρ↑ RMSE↓ per group
    ncols_metric = len(maximize)
    bold = [set() for _ in range(ncols_metric)]
    under = [set() for _ in range(ncols_metric)]
    for col in range(ncols_metric):
        pts = [(i, r[3][col][0], r[3][col][1]) for i, r in enumerate(resolved)
               if not r[2] and r[3][col] is not None and r[3][col][0] is not None]
        if not pts:
            continue
        best = (max if maximize[col] else min)(m for _, m, _ in pts)
        best_std = max(s for _, m, s in pts if abs(m - best) < 1e-9)
        for i, m, s in pts:
            gap = abs(m - best)
            if gap <= best_std and gap <= s:
                bold[col].add(i)
        remaining = [(i, m) for i, m, s in pts if i not in bold[col]]
        if remaining:
            second = (max if maximize[col] else min)(m for _, m in remaining)
            for i, m in remaining:
                if abs(m - second) < 1e-9:
                    under[col].add(i)

    def cell_tex(i, col, pair):
        if pair is None:
            return "TBA"
        if pair[0] is None:               # explicit N/A cell (not-applicable, e.g. zero-shot energy RMSE)
            return "N/A"
        if not maximize[col] and pair[0] >= 100:   # off-scale RMSE (uncalibrated energy) → N/A
            return "N/A"
        text = rf"{pair[0]:.3f}\std{{{pair[1]:.3f}}}"
        if i in bold[col]:
            return rf"\textbf{{{text}}}"
        if i in under[col]:
            return rf"\underline{{{text}}}"
        return text

    lines = []
    i = 0
    for modality, methods in BINDING_MODALITY_GROUPS:
        lines.append(rf"\multirow{{{len(methods)}}}{{*}}{{{modality}}}")
        for display, key, pre, leaked in methods:
            disp_e, _, _, vals = resolved[i]
            if display in BINDING_CMIDRULE_BEFORE:
                lines.append(rf"\cmidrule(lr){{2-{ncols_metric + 3}}}")
            cells = [cell_tex(i, col, vals[col]) for col in range(ncols_metric)]
            lines.append("& " + " & ".join([disp_e, pre, *cells]) + r" \\")
            i += 1
        if modality != BINDING_MODALITY_GROUPS[-1][0]:
            lines.append(r"\midrule")

    header1 = (
        r"\multirow{3}{*}{\textbf{Input}} & \multirow{3}{*}{\textbf{Method}} & "
        r"\multirow{3}{*}{\textbf{Pre.}} & "
        + " & ".join(rf"\multicolumn{{3}}{{c}}{{\textbf{{{h}}}}}" for h in group_headers) + r" \\"
    )
    _mh = r"\textbf{Pearson \textit{r}} & \textbf{Spearman $\rho$} & \textbf{RMSE $\downarrow$}"
    header2 = r"& & & " + " & ".join([_mh] * len(group_headers)) + r" \\"
    body = [
        r"\begin{table}[!t]",
        r"    \centering",
        r"    \caption{",
        f"        {caption}",
        r"    }",
        rf"    \label{{{label}}}",
        r"    \resizebox{.98\textwidth}{!}{%",
        rf"        \begin{{tabular}}{{@{{}}ll{'c' * (ncols_metric + 1)}@{{}}}}",
        r"            \toprule",
        "            " + header1,
        "            " + "".join(rf"\cmidrule(lr){{{4 + 3 * g}-{6 + 3 * g}}}" for g in range(len(group_headers))),
        "            " + header2,
        r"            \midrule",
    ]
    body.extend("            " + ln for ln in lines)
    body.extend([
        r"            \bottomrule",
        r"        \end{tabular}",
        r"    }",
        r"\end{table}",
    ])
    return "\n".join(body)


def binding_affinity_to_latex(table: Tag) -> str:
    return _modality_grouped_to_latex(
        table, (0, 1, 2, 9, 10, 11), BINDING_GROUP_HEADERS,
        BINDING_CAPTION, "tab:result-binding-affinity", expect_metrics=12)


CASF_CAPTION = (
    r"\textbf{CASF-2016 external binding-affinity prediction.} All models are trained on "
    r"LP-PDBBind and evaluated on the held-out CASF-2016 core set, reported over two "
    r"cohorts. Metrics are reported as the mean and standard deviation over five seeds. "
    r"\textbf{Bold} marks the best and \underline{underlining} marks the next-best; methods "
    r"whose intervals overlap are marked together. $\dagger$ denotes methods trained on "
    r"leaked data overlapping with the test set; these are excluded from marking as they "
    r"are not comparable to leakage-controlled methods. N/A denotes unavailable metrics, "
    r"and \textbf{Pre.} denotes whether models use pre-training on external data."
)
CASF_GROUP_HEADERS = ("Core, train-overlap (N=214)", "Non-train (N=124)", "Clean held-out (N=92)")


def _casf_groups(table: Tag):
    """CASF 표의 metric 열 index와 cohort 헤더를 HTML 헤더에서 유도 (모든 cohort 포함).
    Table 1c의 cohort 수(2 또는 3)가 바뀌어도 자동 적응한다."""
    header = table.find("tr")
    cells = header.find_all(["th", "td"], recursive=False)
    indices, headers, start = [], [], 0
    for cell in cells:
        classes = cell.get("class") or []
        if "col-modality" in classes or "col-method" in classes:
            continue
        span = int(cell.get("colspan") or 1)
        text = cell.get_text(" ", strip=True).replace("\xa0", " ")
        n = re.search(r"N\s*=\s*(\d+)", text)
        core = re.sub(r"\s*N\s*=\s*\d+\s*$", "", text.replace("CASF-2016", "")).strip()
        core = (core[:1].upper() + core[1:]) if core else core
        headers.append(rf"{escape_latex_text(core)} (N={n.group(1)})" if n else escape_latex_text(text))
        indices.extend(range(start, start + span))
        start += span
    return tuple(indices), tuple(headers), start


def casf_to_latex(table: Tag) -> str:
    indices, headers, total = _casf_groups(table)
    return _modality_grouped_to_latex(
        table, indices, headers,
        CASF_CAPTION, "tab:result-binding-affinity-casf", expect_metrics=total,
        casf_clean=frozenset({"IPNet (frozen)"}))


def _similarity_groups(table: Tag):
    """similarity-stratified 표의 metric 열 index와 그룹 헤더를 HTML 헤더에서 유도 (LP-PDBBind 열 제외)."""
    header = table.find("tr")
    cells = header.find_all(["th", "td"], recursive=False)
    indices, headers, start = [], [], 0
    for cell in cells:
        classes = cell.get("class") or []
        if "col-modality" in classes or "col-method" in classes:
            continue
        span = int(cell.get("colspan") or 1)
        text = cell.get_text(" ", strip=True).replace("\xa0", " ")
        if "LP-PDBBind" not in text:            # full-test 열은 Table 1과 중복이라 제외
            thr = re.search(r"<\s*(\d+)\s*%", text)
            n = re.search(r"N\s*=\s*(\d+)", text)
            label = (rf"Sequence identity $<${thr.group(1)}\% (N={n.group(1)})"
                     if thr and n else escape_latex_text(text))
            indices.extend(range(start, start + span))
            headers.append(label)
        start += span
    return tuple(indices), tuple(headers), start


def similarity_stratified_to_latex(table: Tag) -> str:
    indices, headers, total = _similarity_groups(table)
    return _modality_grouped_to_latex(
        table, indices, headers, SIMILARITY_MODALITY_CAPTION,
        "tab:result-binding-affinity-seqsim", expect_metrics=total)


def table_to_latex(table: Tag, index: int, keep_badges: bool = True, resize_wide: bool = True) -> str:
    if index == 0:
        return binding_affinity_to_latex(table)
    if index == 1:
        return similarity_stratified_to_latex(table)
    if index == 2:
        return casf_to_latex(table)
    if index == 5:
        return de_novo_to_latex(table)
    rows, occupied, placements, ncols = build_layout(table)
    title = table_title(table, index)
    caption = (
        TABLE1_CAPTION if index == 0
        else SIMILARITY_CAPTION if index == 1
        else caption_from_title(title)
    )
    label = label_from_title(title, index)
    all_alignments = infer_alignments(placements, ncols)
    visible_columns = list(range(ncols))
    custom_headers = {}
    if index == 0:
        custom_headers = dict(TABLE1_GROUP_HEADERS)
        visible_columns = [0, 1] + [
            col
            for start_col, _ in TABLE1_GROUP_HEADERS
            for col in range(start_col, start_col + 3)
        ]
    elif index == 1:
        custom_headers = dict(SIMILARITY_GROUP_HEADERS)
        visible_columns = [0, 1] + [
            col
            for start_col, _ in SIMILARITY_GROUP_HEADERS
            for col in range(start_col, start_col + 3)
        ]
    metric_emphasis = (
        table1_metric_emphasis(rows, placements, set(visible_columns) - {0, 1})
        if index in {0, 1} else {}
    )
    alignments = "".join(all_alignments[col] for col in visible_columns)
    wide = ncols >= 9
    environment = "table*" if wide else "table"

    header_rows = {
        i for i, row in enumerate(rows)
        if row.find_parent("thead") is not None
    }
    if not header_rows and rows and rows[0].find_all("th", recursive=False):
        header_rows = {0}
    last_header_row = max(header_rows) if header_rows else None

    latex_rows = []
    for row_index in range(len(rows)):
        if (
            index in {0, 1}
            and last_header_row is not None
            and row_index > last_header_row + 1
            and rows[row_index].find("td", class_="col-modality", recursive=False)
            is not None
        ):
            latex_rows.append(r"\midrule")
        tokens = []
        visible_index = 0
        while visible_index < len(visible_columns):
            col_index = visible_columns[visible_index]
            placement = occupied.get((row_index, col_index))
            if placement is None:
                tokens.append("")
                visible_index += 1
                continue
            visible_span = [
                col for col in visible_columns
                if placement.col <= col < placement.col + placement.colspan
            ]

            if placement.row == row_index and col_index == visible_span[0]:
                if index in {0, 1} and row_index == 0 and placement.col in custom_headers:
                    header = escape_latex_text(custom_headers[placement.col])
                    value = rf"\textbf{{{header}}}"
                else:
                    if index == 5 and placement.col == 0 and placement.node.name == "td":
                        value = method_name_to_latex(placement.node)
                    else:
                        value = cell_to_latex(
                            placement.node,
                            False
                            if (index == 0 and TABLE1_EXCLUDE_TAGS)
                            or (index == 1 and SIMILARITY_EXCLUDE_TAGS)
                            or (index == 5 and REFERENCE_SIMILARITY_EXCLUDE_TAGS)
                            else keep_badges,
                            preserve_emphasis=(index not in {0, 1} or placement.node.name == "th"),
                        )
                    if index in {0, 1}:
                        value = apply_std_macro(value)
                    emphasis = metric_emphasis.get((row_index, placement.col))
                    if emphasis == "bold":
                        value = rf"\textbf{{{value}}}"
                    elif emphasis == "underline":
                        value = rf"\underline{{{value}}}"
                if placement.rowspan > 1:
                    value = rf"\multirow{{{placement.rowspan}}}{{*}}{{{value}}}"
                visible_colspan = len(visible_span)
                if visible_colspan > 1:
                    value = rf"\multicolumn{{{visible_colspan}}}{{c}}{{{value}}}"
                tokens.append(value)
                visible_index += visible_colspan
            else:
                # 이전 행에서 시작한 multirow가 차지하는 열의 자리표시자입니다.
                span = len(visible_span) if col_index == visible_span[0] else 1
                tokens.append(rf"\multicolumn{{{span}}}{{c}}{{}}" if span > 1 else "")
                visible_index += span

        latex_rows.append(" & ".join(tokens).lstrip() + r" \\ ".rstrip())
        if row_index == last_header_row:
            latex_rows.append(r"\midrule")

    if index in {0, 1}:
        compact_label = (
            "tab:results-binding-affinity"
            if index == 0 else "tab:results-binding-affinity-similarity"
        )
        body = [
            r"\begin{table}[!t]",
            r"    \centering",
            r"    \caption{",
            f"        {caption}",
            r"    }",
            rf"    \label{{{compact_label}}}",
            r"    \resizebox{.98\textwidth}{!}{%",
            rf"        \begin{{tabular}}{{@{{}}{alignments}@{{}}}}",
            r"            \toprule",
        ]
        body.extend("            " + line for line in latex_rows)
        body.extend([
            r"            \bottomrule",
            r"        \end{tabular}",
            r"    }",
            r"\end{table}",
        ])
        return "\n".join(body)

    body = [
        r"% Requires: \usepackage{booktabs,multirow,graphicx}",
        rf"\begin{{{environment}}}[t]",
        r"    \centering",
        r"    \caption{",
        f"        {caption}",
        r"    }",
        rf"    \label{{{label}}}",
    ]
    if wide and resize_wide:
        body.extend([
            r"    \resizebox{.98\textwidth}{!}{%",
            rf"        \begin{{tabular}}{{@{{}}{alignments}@{{}}}}",
            r"            \toprule",
        ])
        body.extend("            " + line for line in latex_rows)
        body.extend([
            r"            \bottomrule",
            r"        \end{tabular}",
            r"    }",
        ])
    else:
        body.extend([
            rf"    \begin{{tabular}}{{@{{}}{alignments}@{{}}}}",
            r"        \toprule",
        ])
        body.extend("        " + line for line in latex_rows)
        body.extend([
            r"        \bottomrule",
            r"    \end{tabular}",
        ])
    body.append(rf"\end{{{environment}}}")
    return "\n".join(body)


# ── de novo Vina table, sourced from results_drug_design.html ──────────────────
def drug_design_vina_to_latex(table: Tag) -> str:
    """Table 1 of results_drug_design.html -> LaTeX.

    That table is the live one for de novo generation, so the Vina numbers are taken
    from it rather than from results.html (whose de novo block is a different, older
    pocket set). Layout: Method + Score/Min/Dock as Avg|Med pairs + High aff. + QED +
    SA + Div + heavy atoms + n, i.e. 13 cells per data row. Section rows carry a single
    cell and are turned into \midrule; rows still reading TBA are emitted commented out
    so the template keeps its shape while a baseline is still sampling.
    """
    body = table.select("tbody > tr")

    def clean(cell: Tag) -> str:
        t = cell.get_text(" ", strip=True)
        t = t.replace("−", "-").replace("—", "---").replace("–", "--")
        return t if t else "---"

    def display_name(cell: Tag) -> str:
        """Method label without the tag chips and sub-line the HTML carries."""
        c = cell.__copy__()
        for junk in c.select(".tag, .sub, .nsub, span[style]"):
            junk.decompose()
        name = c.get_text(" ", strip=True)
        return {
            "Reference ligand": "Reference",
            "VoxBind σ=0.9": r"VoxBind\textsubscript{\scriptsize $\sigma$=0.9}",
            "Ours · v1": r"Ours\textsubscript{\scriptsize v1}",
            "Ours · v2": r"Ours\textsubscript{\scriptsize v2}",
            "Ours · v3": r"Ours\textsubscript{\scriptsize v3}",
        }.get(name, escape_latex_text(name))

    lines, n_tba = [], 0
    for r in body:
        cells = r.find_all(["th", "td"], recursive=False)
        if len(cells) == 1:                       # section divider
            if lines and lines[-1] != r"\midrule":
                lines.append(r"\midrule")
            continue
        if len(cells) != 13:
            continue
        vals = [clean(c) for c in cells[1:]]
        line = " & ".join([display_name(cells[0]), *vals]) + r" \\"
        if any(v == "TBA" for v in vals):         # still sampling
            n_tba += 1
            line = "% " + line
        lines.append(line)

    title = table.find_previous("p", class_="table-title")
    title = title.get_text(" ", strip=True) if title else "Vina affinity"
    caption = (r"\textbf{De novo drug design on CrossDocked.} "
               + escape_latex_text(title.split("·", 1)[-1].strip())
               + r". Vina Score / Min / Dock are reported as Avg\,/\,Med; "
                 r"High aff.\ is the share of molecules out-docking the pocket's "
                 r"reference ligand.")

    header = (
        r"\begin{tabular}{l" + "rr" * 3 + "rrrrrr}" + "\n"
        r"\toprule" + "\n"
        r"\multirow{2}{*}{Method} & \multicolumn{2}{c}{Vina Score $\downarrow$}"
        r" & \multicolumn{2}{c}{Vina Min $\downarrow$}"
        r" & \multicolumn{2}{c}{Vina Dock $\downarrow$}"
        r" & \multirow{2}{*}{\shortstack{High aff.\\ \%\,$\uparrow$}}"
        r" & \multirow{2}{*}{QED\,$\uparrow$} & \multirow{2}{*}{SA\,$\uparrow$}"
        r" & \multirow{2}{*}{Div.\,$\uparrow$}"
        r" & \multirow{2}{*}{\shortstack{Heavy\\ atoms}}"
        r" & \multirow{2}{*}{$n$} \\" + "\n"
        r"\cmidrule(lr){2-3}\cmidrule(lr){4-5}\cmidrule(lr){6-7}" + "\n"
        r" & Avg & Med & Avg & Med & Avg & Med & & & & & & \\" + "\n"
        r"\midrule"
    )
    note = (f"\n% {n_tba} row(s) commented out: still sampling at build time."
            if n_tba else "")
    return (r"\begin{table}[t]" + "\n\\centering\n"
            + r"\caption{" + caption + "}\n"
            + r"\label{tab:denovo-vina}" + "\n"
            + r"\resizebox{.98\textwidth}{!}{%" + "\n"
            + header + "\n" + "\n".join(lines) + "\n"
            + r"\bottomrule" + "\n" + r"\end{tabular}}" + note + "\n"
            + r"\end{table}")


<>:861: SyntaxWarning: invalid escape sequence '\m'
<>:861: SyntaxWarning: invalid escape sequence '\m'
/tmp/ipykernel_1611435/1676435880.py:861: SyntaxWarning: invalid escape sequence '\m'
  cell and are turned into \midrule; rows still reading TBA are emitted commented out


## Results table output

HTML에서 표 목록을 확인하고 `TABLE_INDEX`로 선택한 LaTeX 표를 출력합니다.

In [4]:
resolved_html_path = resolve_html_path(HTML_PATH)
soup = BeautifulSoup(resolved_html_path.read_text(encoding="utf-8"), "html.parser")
tables = soup.find_all("table")

print(f"Source: {resolved_html_path}")
print(f"Found {len(tables)} tables\n")
for index, table in enumerate(tables):
    rows, _, _, ncols = build_layout(table)
    print(f"[{index}] {len(rows)} rows × {ncols} columns | {table_title(table, index)}")


Source: /home1/irteam/VoxBind/notebook/html/results.html
Found 9 tables

[0] 23 rows × 14 columns | Table 1a · Test metrics across cleaning tiers — mean ± std (5 seeds; deterministic zero-shot = 1 pass)
[1] 23 rows × 11 columns | Table 1b · Generalization under protein-sequence novelty (LP: 3 seeds; CL3: fresh 5 seeds; deterministic zero-shot = 1 pass)
[2] 23 rows × 8 columns | Table 1c · CASF-2016 external test — all methods with 90% confidence intervals
[3] 12 rows × 11 columns | Table 1.2 · C vs C+D+G across every affinity test set (ours: 5-seed mean ± std)
[4] 18 rows × 11 columns | Table 3 · CDG-encoder ablation — val ρ / test r / test ρ / RMSE (5 seeds, MSE head)
[5] 17 rows × 19 columns | Table 4 · De novo generation — CrossDocked benchmark
[6] 23 rows × 11 columns | Table A1 · LP-PDBBind protein-novelty cohorts
[7] 23 rows × 11 columns | Table A2 · CASF-2016 additional cohorts — train-overlap core & CL3-protein-novelty
[8] 5 rows × 6 columns | Table B1 · CASP16 chymase (L1000) 

In [5]:
if not 0 <= TABLE_INDEX < len(tables):
    raise IndexError(f"TABLE_INDEX는 0부터 {len(tables) - 1} 사이여야 합니다.")

latex_tables = [
    table_to_latex(table, index, keep_badges=KEEP_BADGES, resize_wide=RESIZE_WIDE)
    for index, table in enumerate(tables)
]
latex = latex_tables[TABLE_INDEX]
print(latex)

# 필요하면 아래 두 줄의 주석을 풀어 .tex 파일로도 저장할 수 있습니다.
# output_path = Path(f"results_table_{TABLE_INDEX + 1}.tex")
# output_path.write_text(latex + "\n", encoding="utf-8")


\begin{table}[!t]
    \centering
    \caption{
        \textbf{Structure-based drug design results} on 92 test pockets with experimental density data available of CrossDocked2020 benchmark. Vina and high-affinity cells report mean / median; arrows indicate the preferred direction.
    }
    \label{tab:result-drug-design}
    \resizebox{.98\textwidth}{!}{%
        \begin{tabular}{@{}lcccccccc@{}}
            \toprule
            \multirow{2}{*}{\textbf{Method}} & \multicolumn{4}{c}{\textbf{Vina evaluation}} & \multicolumn{4}{c}{\textbf{Sample quality}} \\
            \cmidrule(lr){2-5}\cmidrule(lr){6-9}
            & \textbf{Score $\downarrow$} & \textbf{Min $\downarrow$} & \textbf{Dock $\downarrow$} & \textbf{High aff. $\uparrow$} & \textbf{QED $\uparrow$} & \textbf{SA $\uparrow$} & \textbf{Diversity $\uparrow$} & \textbf{\# Atoms / Mol} \\
            \midrule
            Reference & -6.36 / -6.46 & -6.71 / -6.49 & -7.45 / -7.26 & --- / --- & 0.48 & 0.73 & --- \\
            \midrule


## CL1 and CL2 filtered table output

Test/CL3 형식을 재사용해 CL1 및 CL2 filtering 결과를 각각 출력합니다.

In [6]:
def binding_affinity_cl1_cl2_to_latex(table: Tag) -> str:
    """CL1/CL2-filtered tiers of the binding-affinity table, in the modality (Pre. 열) format."""
    return _modality_grouped_to_latex(
        table, (3, 4, 5, 6, 7, 8),
        ("CL1 filtered (N=1166)", "CL2 filtered (N=1149)"),
        BINDING_CL1_CL2_CAPTION, "tab:result-binding-affinity-cl1-cl2", expect_metrics=12)


cl1_cl2_latex = binding_affinity_cl1_cl2_to_latex(tables[0])
print(cl1_cl2_latex)

\begin{table}[!t]
    \centering
    \caption{
        \textbf{LP-PDBBind binding-affinity prediction on the \textit{CL1-filtered} and \textit{CL2-filtered} test sets.} Metrics are reported as the mean and standard deviation over five seeds. \textbf{Bold} marks the best and \underline{underlining} marks the next-best; methods whose intervals overlap are marked together. $\dagger$ denotes methods trained on leaked data overlapping with the test set; these are excluded from marking as they are not comparable to leakage-controlled methods. N/A denotes unavailable metrics, and \textbf{Pre.} denotes whether models use pre-training on external data.
    }
    \label{tab:result-binding-affinity-cl1-cl2}
    \resizebox{.98\textwidth}{!}{%
        \begin{tabular}{@{}llccccccc@{}}
            \toprule
            \multirow{3}{*}{\textbf{Input}} & \multirow{3}{*}{\textbf{Method}} & \multirow{3}{*}{\textbf{Pre.}} & \multicolumn{3}{c}{\textbf{CL1 filtered (N=1166)}} & \multicolumn{3}{c}{\textbf{CL2

## CASF clean held-out table (with 90% CI column)

`tables[2]`의 clean-92 코호트만 뽑아 Pearson r 옆에 BCa 90% CI 열을 추가한 표입니다 (mean±std·bold·underline은 그대로).

In [7]:
# ── CASF-2016 clean-92 focused table: Pearson r with a dedicated 90% CI column ──
# New table type (same pattern as the CL1/CL2 table above). Reads the CASF Table 1c
# (tables[2], non-train + clean), keeps ONLY the clean-92 cohort, and splits the BCa
# 90% CI into its own column. std parsed from the '±' part only (CI is separate).

CASF_ABS = frozenset({"DSMBind"})   # zero-shot |r| — excluded from ranking (matches HTML unranked_ctx)
_NUM = r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)"


def _casf_parse_metric_ci(cell):
    """(mean, std, (lo,hi)|None). std ONLY from the '±...' part (not the CI); CI from '[lo, hi]'."""
    tbd = cell.find(class_="tbd")
    if tbd is not None and tbd.get_text(strip=True).lower() in ("n/a", "na"):
        return (None, None, None)
    val = cell.find(class_="val")
    if val is None:
        return None
    mean = float(re.search(_NUM, val.get_text(strip=True).replace("−", "-")).group())
    std, ci = 0.0, None
    sd = cell.find(class_="sd")
    if sd:
        txt = sd.get_text(" ", strip=True).replace("\xa0", " ")
        sm = re.search(r"±\s*(" + _NUM + ")", txt)
        if sm:
            std = abs(float(sm.group(1)))
        cm = re.search(r"\[\s*(" + _NUM + r")\s*,\s*(" + _NUM + r")\s*\]", txt)
        if cm:
            ci = (float(cm.group(1)), float(cm.group(2)))
    return (mean, std, ci)


def casf_clean_ci_to_latex(table):
    indices, headers, total = _casf_groups(table)
    clean_g = next((gi for gi, h in enumerate(headers) if re.search(r"clean|held", h, re.I)), len(headers) - 1)
    clean_idx = list(indices[3 * clean_g:3 * clean_g + 3])

    html_rows = []
    for row in table.find_all("tr"):
        method_cell = row.find("td", class_="col-method", recursive=False)
        metrics = row.find_all("td", class_="metric", recursive=False)
        if method_cell is None or len(metrics) != total:
            continue
        text = method_cell.get_text(" ", strip=True)
        html_rows.append((text, [_casf_parse_metric_ci(metrics[i]) for i in clean_idx]))

    casf_clean = frozenset({"IPNet (frozen)"})
    resolved = []
    for _, methods in BINDING_MODALITY_GROUPS:
        for display, key, pre, leaked in methods:
            match = next((v for text, v in html_rows if text.startswith(key)), None)
            if match is None:
                raise ValueError(f"clean-CASF: {key!r} 행 없음")
            disp_e, leaked_e = ((display.replace(r"$^\dagger$", ""), False)
                                if key in casf_clean else (display, leaked))
            excluded = leaked_e or key in CASF_ABS      # not eligible for the best-highlight
            resolved.append((disp_e, pre, excluded, match))

    maximize = [True, True, False]
    bold = [set() for _ in range(3)]
    under = [set() for _ in range(3)]
    for col in range(3):
        pts = [(i, r[3][col][0], r[3][col][1]) for i, r in enumerate(resolved)
               if not r[2] and r[3][col] is not None and r[3][col][0] is not None]
        if not pts:
            continue
        best = (max if maximize[col] else min)(m for _, m, _ in pts)
        reaches = ((lambda m, s: m + s >= best - 1e-12) if maximize[col]
                   else (lambda m, s: m - s <= best + 1e-12))
        for i, m, s in pts:
            if reaches(m, s):
                bold[col].add(i)
        rest = [(i, m) for i, m, s in pts if i not in bold[col]]
        if rest:
            sec = (max if maximize[col] else min)(m for _, m in rest)
            for i, m in rest:
                if abs(m - sec) < 1e-9:
                    under[col].add(i)

    def mcell(i, col, tr):
        if tr is None:
            return "TBA"
        m, s, _ = tr
        if m is None:
            return "N/A"
        if not maximize[col] and m >= 100:
            return "N/A"
        t = rf"{m:.3f}\std{{{s:.3f}}}" if s else rf"{m:.3f}"   # hide ±0.000 (single-vector)
        if i in bold[col]:
            return rf"\textbf{{{t}}}"
        if i in under[col]:
            return rf"\underline{{{t}}}"
        return t

    def cicell(tr):
        if tr is None or tr[0] is None or tr[2] is None:
            return r"\textendash"
        lo, hi = tr[2]
        return rf"{{\scriptsize $[{lo:.2f},\,{hi:.2f}]$}}"

    lines = []
    i = 0
    for modality, methods in BINDING_MODALITY_GROUPS:
        lines.append(rf"\multirow{{{len(methods)}}}{{*}}{{{modality}}}")
        for display, key, pre, leaked in methods:
            vals = resolved[i][3]
            cells = [resolved[i][0], pre,
                     mcell(i, 0, vals[0]), cicell(vals[0]),
                     mcell(i, 1, vals[1]), cicell(vals[1]),
                     mcell(i, 2, vals[2]), cicell(vals[2])]
            lines.append("& " + " & ".join(cells) + r" \\")
            i += 1
        if modality != BINDING_MODALITY_GROUPS[-1][0]:
            lines.append(r"\midrule")

    caption = (
        r"\textbf{CASF-2016 clean held-out (N=92) binding-affinity prediction.} "
        r"All models are trained on LP-PDBBind and evaluated on the CASF-2016 core complexes held out "
        r"from both our train and validation splits. Pearson \textit{r} / Spearman $\rho$ / RMSE are the "
        r"5-seed mean\,$\pm$\,std, each followed by its \textbf{90\% CI} --- a BCa bootstrap that resamples the "
        r"test complexes and averages the per-seed metric, matching CASF-2016 \S2.4. "
        r"Methods shown \emph{without} $\pm$std are single deterministic models (no training seed) --- the "
        r"zero-shot scorers Nesso-1 and DSMBind; for these the interval is pure test-set sampling. "
        r"\textbf{Bold} marks results tied with the best (error bar reaches the best mean); "
        r"\underline{underline} marks the runner-up. $^\dagger$: affinity-leaked reference.")
    header1 = (r"\multirow{2}{*}{\textbf{Input}} & \multirow{2}{*}{\textbf{Method}} & "
               r"\multirow{2}{*}{\textbf{Pre.}} & \multicolumn{6}{c}{\textbf{CASF-2016 clean held-out (N\,=\,92)}} \\")
    header2 = (r"& & & \textbf{Pearson \textit{r}} & \textbf{90\% CI} & "
               r"\textbf{Spearman $\rho$} & \textbf{90\% CI} & "
               r"\textbf{RMSE $\downarrow$} & \textbf{90\% CI} \\")
    body = [
        r"\begin{table}[!t]",
        r"    \centering",
        r"    \caption{",
        f"        {caption}",
        r"    }",
        r"    \label{tab:result-casf-clean}",
        r"    \resizebox{.98\textwidth}{!}{%",
        r"        \begin{tabular}{@{}llccccccc@{}}",
        r"            \toprule",
        "            " + header1,
        r"            \cmidrule(lr){4-9}",
        "            " + header2,
        r"            \midrule",
    ]
    body.extend("            " + ln for ln in lines)
    body.extend([
        r"            \bottomrule",
        r"        \end{tabular}",
        r"    }",
        r"\end{table}",
    ])
    return "\n".join(body)


casf_clean_latex = casf_clean_ci_to_latex(tables[2])
print(casf_clean_latex)


\begin{table}[!t]
    \centering
    \caption{
        \textbf{CASF-2016 clean held-out (N=92) binding-affinity prediction.} All models are trained on LP-PDBBind and evaluated on the CASF-2016 core complexes held out from both our train and validation splits. Pearson \textit{r} / Spearman $\rho$ / RMSE are the 5-seed mean\,$\pm$\,std, each followed by its \textbf{90\% CI} --- a BCa bootstrap that resamples the test complexes and averages the per-seed metric, matching CASF-2016 \S2.4. Methods shown \emph{without} $\pm$std are single deterministic models (no training seed) --- the zero-shot scorers Nesso-1 and DSMBind; for these the interval is pure test-set sampling. \textbf{Bold} marks results tied with the best (error bar reaches the best mean); \underline{underline} marks the runner-up. $^\dagger$: affinity-leaked reference.
    }
    \label{tab:result-casf-clean}
    \resizebox{.98\textwidth}{!}{%
        \begin{tabular}{@{}llccccccc@{}}
            \toprule
            \multirow{2}

## De novo Vina table (`results_drug_design.html`)

de novo 생성 결과의 Vina 표는 `results.html`이 아니라 **`results_drug_design.html`의 Table 1**에서 가져옵니다. 
그쪽이 현재 살아 있는 78-포켓 평가이고, `results.html`의 de novo 블록은 포켓 세트가 다른 예전 표입니다.

아직 샘플링 중이라 `TBA`인 행은 주석 처리되어 출력되므로, baseline이 끝나는 대로 이 셀만 다시 실행하면 값이 채워집니다.

In [8]:
# Vina 결과 전용 소스: results_drug_design.html
DRUG_DESIGN_HTML_PATH = Path("html/results_drug_design.html")
DRUG_DESIGN_VINA_TITLE = "Vina affinity"   # 이 문구를 table-title에 포함하는 표를 고릅니다

dd_path = resolve_html_path(DRUG_DESIGN_HTML_PATH)
dd_soup = BeautifulSoup(dd_path.read_text(encoding="utf-8"), "html.parser")

def _dd_title(tbl):
    t = tbl.find_previous("p", class_="table-title")
    return t.get_text(" ", strip=True) if t else ""

dd_tables = dd_soup.find_all("table")
matches = [t for t in dd_tables if DRUG_DESIGN_VINA_TITLE in _dd_title(t)]
if not matches:
    raise LookupError(
        f"{DRUG_DESIGN_VINA_TITLE!r} 를 제목에 담은 표가 없습니다. 후보:\n  "
        + "\n  ".join(f"[{i}] {_dd_title(t)}" for i, t in enumerate(dd_tables))
    )

print(f"Source: {dd_path}")
print(f"Matched: {_dd_title(matches[0])}\n")
print(drug_design_vina_to_latex(matches[0]))

Source: /home1/irteam/VoxBind/notebook/html/results_drug_design.html
Matched: Table 1  ·  Vina affinity and sample quality — 79 pockets, whole receptor

\begin{table}[t]
\centering
\caption{\textbf{De novo drug design on CrossDocked.} Vina affinity and sample quality --- 79 pockets, whole receptor. Vina Score / Min / Dock are reported as Avg\,/\,Med; High aff.\ is the share of molecules out-docking the pocket's reference ligand.}
\label{tab:denovo-vina}
\resizebox{.98\textwidth}{!}{%
\begin{tabular}{lrrrrrrrrrrrr}
\toprule
\multirow{2}{*}{Method} & \multicolumn{2}{c}{Vina Score $\downarrow$} & \multicolumn{2}{c}{Vina Min $\downarrow$} & \multicolumn{2}{c}{Vina Dock $\downarrow$} & \multirow{2}{*}{\shortstack{High aff.\\ \%\,$\uparrow$}} & \multirow{2}{*}{QED\,$\uparrow$} & \multirow{2}{*}{SA\,$\uparrow$} & \multirow{2}{*}{Div.\,$\uparrow$} & \multirow{2}{*}{\shortstack{Heavy\\ atoms}} & \multirow{2}{*}{$n$} \\
\cmidrule(lr){2-3}\cmidrule(lr){4-5}\cmidrule(lr){6-7}
 & Avg & Med & Avg & 